> 在线讲义：[第 66 章 · 跨设备分工与会话迁移：传图片、传特征，还是传状态](https://wikieden.github.io/ai-infra-from-constraints/lessons/66-cross-device.html) · 本书《从约束推导 AI Infra》

# 第 66 章 · 跨设备分工与会话迁移：传图片、传特征，还是传状态

配套讲义：仓库里 `ai-infra/notes/lessons/66-cross-device.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



端侧编码后上传特征、会话迁移的回本轮数、跨设备张量并行的启动开销，取自 AII 第 12 章 §12.2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MiB = 2**20

**函数 `feature_bytes`**：最终投影 + 三组 DeepStack，每组同宽。

In [ ]:
def feature_bytes(tokens: int = 400, width: int = 2560, groups: int = 4, b: int = 2) -> int:
    """最终投影 + 三组 DeepStack，每组同宽。"""
    return tokens * width * groups * b

**函数 `encode_delta`**：ΔT = T_E,l − T_E,r + T_x + (S_E − S_I)/B_u；< 0 时端侧编码更快。

In [ ]:
def encode_delta(T_el: float, T_er: float, Tx: float, S_E: float, S_I: float, Bu: float) -> float:
    """ΔT = T_E,l − T_E,r + T_x + (S_E − S_I)/B_u；< 0 时端侧编码更快。"""
    return T_el - T_er + Tx + (S_E - S_I) / Bu

**函数 `migrate_cost`**：小工具。

In [ ]:
def migrate_cost(state_bytes: float, bw: float, restore: float) -> float:
    return state_bytes * 8 / bw + restore

**函数 `gain`**：G(N) = N Δt − M。

In [ ]:
def gain(N: int, dt: float, M: float) -> float:
    """G(N) = N Δt − M。"""
    return N * dt - M

**函数 `breakeven_rounds`**：小工具。

In [ ]:
def breakeven_rounds(dt: float, M: float) -> int:
    n = 1
    while gain(n, dt, M) <= 0:
        n += 1
    return n

**函数 `tp_startup_ms`**：每层两次归约 × 每次两阶段启动 = 144α。

In [ ]:
def tp_startup_ms(layers: int = 36, alpha_us: float = 2) -> float:
    """每层两次归约 × 每次两阶段启动 = 144α。"""
    return layers * 2 * 2 * alpha_us / 1000

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：上传图片还是上传特征")
S_E, S_I = feature_bytes(), 0.8e6
print(f"  特征 400 × 10240 × 2 = {S_E:,} B（{S_E / MiB:.1f} MiB），截图 0.8 MB；6.4 Mbit/s 上传 {S_E * 8 / 6.4e6:.1f} s vs {S_I * 8 / 6.4e6:.1f} s")
le, re = 1.31e12 / 165.2e12, 1.31e12 / 989.4e12
print(f"  编码 1.31 TFLOPs：4090 {le * 1e3:.1f} ms、H100 {re * 1e3:.1f} ms；ΔT = {encode_delta(le, re, 0, S_E, S_I, 0.8e6):.2f} s（端侧更慢）")
print(f"  同机房 50 GB/s：传输差 {(S_E - S_I) / 50e9 * 1e3:.2f} ms")
print(f"  400 个视觉 token 的语言 KV：{400 * 147456 / MiB:.1f} MiB")

In [ ]:
print("\n第二步：会话迁移")
M = migrate_cost(64 * MiB, 80e6, 1)
print(f"  64 MiB 经 80 Mbit/s + 恢复 1 s：M = {M:.1f} s")
for n in (10, 20, 40):
    print(f"  剩 {n} 轮、每轮快 0.4 s：{gain(n, 0.4, M):+.1f} s")
print(f"  回本 {breakeven_rounds(0.4, M)} 轮；恢复再多 1 s → {breakeven_rounds(0.4, M + 1)} 轮")

In [ ]:
print("\n第三步：模型内部切分")
print(f"  TP = 2、36 层：144α；α = 2 μs → {tp_startup_ms():.2f} ms；Wi-Fi 短帧交换 134 μs → {tp_startup_ms(alpha_us=134):.1f} ms（RTX PRO 6000 一步读取下界 9.12 ms）")

## 3. 代码里才看得到的

1. `feature_bytes` 的 10240 = 2560 × 4 组；只算最终投影会漏掉 DeepStack 的三组。
2. 400 个视觉 token 的语言 KV 按 147,456 B/token 是 56.25 MiB，原文写 56.3。
3. `breakeven_rounds` 用未取整的 M（7.71 s），所以 19 轮还差一点，20 轮才回本。

## 4. 自己改一改

- 给 `encode_delta` 加 EC 命中率 h：多轮任务里只有未命中才要编码和传输。
- 用 `gain` 找"每轮快 0.2 s"时的回本轮数。
- 把 `tp_startup_ms` 改成流水线并行：每个 token 只在两阶段间传一次激活。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 压缩包小要解压，文件夹大不用解，慢路上寄压缩包；
> 搬家花一天，每天省半小时，住够久才值；
> 合写书每段都打电话，电话再短也受不了。